<a href="https://colab.research.google.com/github/briskicedteaa/DopaGen/blob/main/Notebooks/DopaGen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install biopython
from Bio import SeqIO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 16.4 MB/s  0:00:00


In [2]:
import zipfile
import os

zip_paths = [
    "/content/D1Receptor.zip",
    "/content/D2Receptor.zip",
    "/content/D3Receptor.zip",
    "/content/D4Receptor.zip",
    "/content/D5Receptor.zip"
]

extract_folder = "/content/PDB"

os.makedirs(extract_folder, exist_ok=True)

for path in zip_paths:
    if os.path.exists(path):
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

In [3]:
zip_paths1 = [
    "/content/COMT.zip",
    "/content/DAT.zip",
    "/content/DDC.zip",
    "/content/MAOB.zip",
    "/content/VMAT.zip"
]

extract_folder = "/content/PDB1"

os.makedirs(extract_folder, exist_ok=True)

for path in zip_paths1:
    if os.path.exists(path):
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

In [4]:
zip_paths2 = [
    "/content/CatecholO.zip",
    "/content/DA-Transpoer.zip",
    "/content/MOA-A.zip"
]

extract_folder = "/content/PDB2"

os.makedirs(extract_folder, exist_ok=True)

for path in zip_paths2:
    if os.path.exists(path):
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

In [5]:
zip_paths3 = [
    "/content/5-HT2A.zip",
    "/content/AdenosineA2A.zip",
    "/content/NTSR1.zip",
    "/content/GPCR88.zip"
]

extract_folder = "/content/PDB3"

os.makedirs(extract_folder, exist_ok=True)

for path in zip_paths3:
    if os.path.exists(path):
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

In [6]:
import pandas as pd

files = [
    "/content/PDB/D1.csv",
    "/content/PDB/D2.csv",
    "/content/PDB/D3.csv",
    "/content/PDB/D4.csv",
    "/content/PDB/D5.csv",
    "/content/PDB1/COMT.csv",
    "/content/PDB1/DAT.csv",
    "/content/PDB1/DDC.csv",
    "/content/PDB1/MAOB.csv",
    "/content/PDB2/CatecholO.csv",
    "/content/PDB2/DA-Transporter.csv",
    "/content/PDB2/MOA-A.csv",
    "/content/PDB3/5-HT2A.csv",
    "/content/PDB3/AdenosineA2A.csv",
    "/content/PDB3/NTSR1.csv",
    "/content/PDB3/GPCR88.csv"
]

df = pd.concat([pd.read_csv(f, sep=";") for f in files], ignore_index=True)

FileNotFoundError: [Errno 2] No such file or directory: '/content/PDB/D1.csv'

In [ ]:
features = [
    "Molecule ChEMBL ID",
    "Smiles",
    "Target ChEMBL ID",
    "Action Type",
]

df = df[features].copy()
df.head()

In [ ]:
df = df[features].dropna().copy()

df = df.drop_duplicates(subset=["Molecule ChEMBL ID"]).copy()

print(df["Action Type"].value_counts())
print("Remaining Ligands:", len(df))
df.groupby("Molecule ChEMBL ID")["Action Type"].nunique().value_counts()
df.groupby("Molecule ChEMBL ID").nunique().value_counts()

In [ ]:
df.groupby(["Target ChEMBL ID", "Action Type"]).size().unstack(fill_value=0)

In [ ]:
df.to_csv('/content/ligands_dataset.csv', index=False)
print('Saved df to /content/ligands_dataset.csv')

In [9]:
import pandas as pd

df = pd.read_csv("/content/ligands_dataset.csv")

df = df[
    ~df["Action Type"].str.contains(
        "ACTIVATOR",
        case=False,
        na=False
    )
].copy()

df = df[
    ~df["Action Type"].str.contains(
        "INVERSE",
        case=False,
        na=False
    )
].copy()

df = df[
    ~df["Action Type"].str.contains(
        "DISRUPTING",
        case=False,
        na=False
    )
].copy()

df = df[
    ~df["Action Type"].str.contains(
        "SUBSTRATE",
        case=False,
        na=False
    )
].copy()

df = df[df["Action Type"] != "MODULATOR"].copy()

print(df["Action Type"].value_counts())
print("Remaining Ligands:", len(df))
df.groupby("Molecule ChEMBL ID")["Action Type"].nunique().value_counts()
df.groupby("Molecule ChEMBL ID").nunique().value_counts()

Action Type
INHIBITOR                        2072
ANTAGONIST                       1199
AGONIST                           816
BINDING AGENT                     769
PARTIAL AGONIST                   135
POSITIVE ALLOSTERIC MODULATOR      57
NEGATIVE ALLOSTERIC MODULATOR      46
Name: count, dtype: int64
Remaining Ligands: 5094


,,,count
Smiles,Target ChEMBL ID,Action Type,
1,1,1,5094


In [10]:
import sys
!{sys.executable} -m pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 8.0 MB/s  0:00:03


In [11]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def get_morgan_fp(smiles, radius=2, n_bits=2048):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            arr = np.zeros((1,), dtype=np.int8)
            Chem.DataStructs.ConvertToNumpyArray(fp, arr)
            return arr
    except:
        pass
    return None

df['Morgan_Fingerprint'] = df['Smiles'].apply(get_morgan_fp)
display(df[['Molecule ChEMBL ID', 'Smiles', 'Morgan_Fingerprint']].head())

,Molecule ChEMBL ID,Smiles,Morgan_Fingerprint
0,CHEMBL5179930,O=C(O)C(=O)O.O=C1CCc2ccccc2N1CCCN1CCN(c2ccc(Cl...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,CHEMBL5169993,O=C(O)C(=O)O.c1ccc(N2CCN(CCCN3CCCc4ccccc43)CC2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,CHEMBL5196870,O=C(O)C(=O)O.O=C1CCc2ccccc2N1CCCN1CCN(c2ccc(Cl...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,CHEMBL267014,Clc1ccc(N2CCN(Cc3c[nH]c4ncccc34)CC2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,CHEMBL5172201,O=C(O)C(=O)O.O=C1CCc2ccccc2N1CCCN1CCN(c2ccccc2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [12]:
from sklearn.model_selection import GroupShuffleSplit

X = np.array(df["Morgan_Fingerprint"].tolist())

y = df["Target ChEMBL ID"].values

groups = df["Molecule ChEMBL ID"].values

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4075
Testing samples: 1019


In [13]:
from sklearn.ensemble import RandomForestClassifier

target_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

target_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

In [14]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = target_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8842001962708538

Classification Report:
               precision    recall  f1-score   support

   CHEMBL1850       0.67      1.00      0.80         2
   CHEMBL1951       0.69      0.41      0.51        27
   CHEMBL2023       1.00      0.80      0.89         5
   CHEMBL2039       0.85      0.98      0.91       322
   CHEMBL2056       0.86      0.68      0.76        37
    CHEMBL217       0.91      0.85      0.88        88
    CHEMBL219       0.83      0.76      0.79        94
    CHEMBL224       0.89      0.93      0.91       145
    CHEMBL234       0.89      0.84      0.86        75
    CHEMBL238       1.00      0.41      0.58        27
    CHEMBL251       0.95      0.96      0.95       160
CHEMBL3399910       1.00      1.00      1.00        26
   CHEMBL4123       1.00      1.00      1.00        11

     accuracy                           0.88      1019
    macro avg       0.89      0.82      0.83      1019
 weighted avg       0.89      0.88      0.88      1019



In [15]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=target_model.classes_
)

cm_df = pd.DataFrame(
    cm,
    index=target_model.classes_,
    columns=target_model.classes_
)

print(cm_df)

               CHEMBL1843  CHEMBL1850  CHEMBL1951  CHEMBL2023  CHEMBL2039  \
CHEMBL1843              0           0           0           0           0   
CHEMBL1850              0           2           0           0           0   
CHEMBL1951              0           0          11           0          15   
CHEMBL2023              0           0           0           4           0   
CHEMBL2039              0           0           5           0         314   
CHEMBL2056              0           0           0           0           4   
CHEMBL217               0           0           0           0           5   
CHEMBL219               0           0           0           0           8   
CHEMBL224               0           1           0           0           5   
CHEMBL234               0           0           0           0           2   
CHEMBL238               0           0           0           0           9   
CHEMBL251               0           0           0           0           6   

In [16]:
target_encoded = pd.get_dummies(df["Target ChEMBL ID"])

X_action = np.hstack([X, target_encoded.to_numpy()])

y_action = df["Action Type"].values

X_action_train = X_action[train_idx]
X_action_test = X_action[test_idx]

y_action_train = y_action[train_idx]
y_action_test = y_action[test_idx]

action_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

action_model.fit(X_action_train, y_action_train)

print("Model trained.")

Model trained.


In [17]:
y_action_pred = action_model.predict(X_action_test)

print("Accuracy:", accuracy_score(y_action_test, y_action_pred))
print(classification_report(y_action_test, y_action_pred, zero_division=0))

Accuracy: 0.9254170755642787
                               precision    recall  f1-score   support

                      AGONIST       0.91      0.90      0.90       173
                   ANTAGONIST       0.91      0.90      0.90       224
                BINDING AGENT       0.89      0.85      0.87       153
                    INHIBITOR       0.95      0.99      0.97       424
NEGATIVE ALLOSTERIC MODULATOR       1.00      0.83      0.91         6
              PARTIAL AGONIST       0.96      0.77      0.85        30
POSITIVE ALLOSTERIC MODULATOR       1.00      0.78      0.88         9

                     accuracy                           0.93      1019
                    macro avg       0.94      0.86      0.90      1019
                 weighted avg       0.93      0.93      0.92      1019

